In [9]:
# Cell 1 — Title
# ============================================================
# NOTEBOOK 03 — PREPROCESSING
# Stage 3: Memory Feature Collection Layer (cleaning phase)
# ============================================================

In [10]:
# Cell 2 — Imports
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import joblib
import os

RAW_PATH  = os.path.abspath(os.path.join(os.getcwd(), '..', 'Data', 'raw', 'Obfuscated-MalMem2022.csv'))
PROC_DIR  = os.path.abspath(os.path.join(os.getcwd(), '..', 'Data', 'processed'))
MODEL_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'models'))

os.makedirs(PROC_DIR,  exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

df = pd.read_csv(RAW_PATH)
print(f"Loaded: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Loaded: (58596, 57)
Columns: ['Category', 'pslist.nproc', 'pslist.nppid', 'pslist.avg_threads', 'pslist.nprocs64bit', 'pslist.avg_handlers', 'dlllist.ndlls', 'dlllist.avg_dlls_per_proc', 'handles.nhandles', 'handles.avg_handles_per_proc', 'handles.nport', 'handles.nfile', 'handles.nevent', 'handles.ndesktop', 'handles.nkey', 'handles.nthread', 'handles.ndirectory', 'handles.nsemaphore', 'handles.ntimer', 'handles.nsection', 'handles.nmutant', 'ldrmodules.not_in_load', 'ldrmodules.not_in_init', 'ldrmodules.not_in_mem', 'ldrmodules.not_in_load_avg', 'ldrmodules.not_in_init_avg', 'ldrmodules.not_in_mem_avg', 'malfind.ninjections', 'malfind.commitCharge', 'malfind.protection', 'malfind.uniqueInjections', 'psxview.not_in_pslist', 'psxview.not_in_eprocess_pool', 'psxview.not_in_ethread_pool', 'psxview.not_in_pspcid_list', 'psxview.not_in_csrss_handles', 'psxview.not_in_session', 'psxview.not_in_deskthrd', 'psxview.not_in_pslist_false_avg', 'psxview.not_in_eprocess_pool_false_avg', 'psxview.n

In [11]:
# Cell 3 — Drop constant columns
constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
print(f"Constant columns found: {constant_cols}")
df.drop(columns=constant_cols, inplace=True, errors='ignore')
print(f"Shape after dropping constants: {df.shape}")

Constant columns found: ['pslist.nprocs64bit', 'handles.nport', 'svcscan.interactive_process_services']
Shape after dropping constants: (58596, 54)


In [12]:
# Cell 4 — Handle missing and infinite values
numeric_cols = df.select_dtypes(include=[np.number]).columns

# Replace inf values first
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Fill missing values with median
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

print(f"Missing values remaining: {df.isnull().sum().sum()}")
print(f"Shape: {df.shape}")

Missing values remaining: 0
Shape: (58596, 54)


In [13]:
# Cell 5 — Check label columns
print("Class column unique values  :", df['Class'].unique())
print("Category column unique values:", df['Category'].unique())
print("\nClass distribution:")
print(df['Class'].value_counts())

Class column unique values  : ['Benign' 'Malware']
Category column unique values: ['Benign'
 'Ransomware-Ako-00a2c6bab1e53f679cdd4fdc772cd291928c109b9b747652639a1700d844f719-1.raw'
 'Ransomware-Ako-00a2c6bab1e53f679cdd4fdc772cd291928c109b9b747652639a1700d844f719-10.raw'
 ...
 'Ransomware-Shade-faddeea111a25da4d0888f3044ae9555f0c55517f6226b30e521008fceda6bbf-7.raw'
 'Ransomware-Shade-f866c086af2e1d8ebaa6f2c8631578896768285120b57ddd43453bdebb217ab1-10.raw'
 'Ransomware-Shade-955d9af38346c1755527bd196668edfad6d3f001d217b04d2380eb99e0760585-8.raw']

Class distribution:
Class
Benign     29298
Malware    29298
Name: count, dtype: int64


In [14]:
# Cell 6 — Separate features from labels
label_cols = ['Category', 'Class']
X = df.drop(columns=label_cols)
y = df['Class']

print(f"Features (X) shape : {X.shape}")
print(f"Labels   (y) shape : {y.shape}")
print(f"Feature names      : {X.columns.tolist()}")

Features (X) shape : (58596, 52)
Labels   (y) shape : (58596,)
Feature names      : ['pslist.nproc', 'pslist.nppid', 'pslist.avg_threads', 'pslist.avg_handlers', 'dlllist.ndlls', 'dlllist.avg_dlls_per_proc', 'handles.nhandles', 'handles.avg_handles_per_proc', 'handles.nfile', 'handles.nevent', 'handles.ndesktop', 'handles.nkey', 'handles.nthread', 'handles.ndirectory', 'handles.nsemaphore', 'handles.ntimer', 'handles.nsection', 'handles.nmutant', 'ldrmodules.not_in_load', 'ldrmodules.not_in_init', 'ldrmodules.not_in_mem', 'ldrmodules.not_in_load_avg', 'ldrmodules.not_in_init_avg', 'ldrmodules.not_in_mem_avg', 'malfind.ninjections', 'malfind.commitCharge', 'malfind.protection', 'malfind.uniqueInjections', 'psxview.not_in_pslist', 'psxview.not_in_eprocess_pool', 'psxview.not_in_ethread_pool', 'psxview.not_in_pspcid_list', 'psxview.not_in_csrss_handles', 'psxview.not_in_session', 'psxview.not_in_deskthrd', 'psxview.not_in_pslist_false_avg', 'psxview.not_in_eprocess_pool_false_avg', 'psxvi

In [15]:
# Cell 7 — Scale features AND create X_scaled_df in same cell
# IMPORTANT: Both scaler.fit_transform and X_scaled_df creation
# happen here together so the variable is always available

scaler     = MinMaxScaler()
X_scaled   = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print(f"Scaling complete.")
print(f"X_scaled_df shape : {X_scaled_df.shape}")
print(f"Value range — min : {X_scaled_df.values.min():.4f}")
print(f"Value range — max : {X_scaled_df.values.max():.4f}")
print(f"\nFirst 3 rows of scaled data:")
X_scaled_df.head(3)

Scaling complete.
X_scaled_df shape : (58596, 52)
Value range — min : 0.0000
Value range — max : 1.0000

First 3 rows of scaled data:


,pslist.nproc,pslist.nppid,pslist.avg_threads,pslist.avg_handlers,dlllist.ndlls,dlllist.avg_dlls_per_proc,handles.nhandles,handles.avg_handles_per_proc,handles.nfile,handles.nevent,...,modules.nmodules,svcscan.nservices,svcscan.kernel_drivers,svcscan.fs_drivers,svcscan.process_services,svcscan.shared_process_services,svcscan.nactive,callbacks.ncallbacks,callbacks.nanonymous,callbacks.ngeneric
0,0.109589,0.140625,0.587121,0.006766,0.369275,0.679940,0.005379,0.004187,0.000501,0.316922,...,1.000000,0.980066,0.994012,1.0,0.85,0.978261,0.919192,0.948718,0.0,1.0
1,0.118721,0.171875,0.651490,0.008354,0.506311,0.802714,0.007541,0.005075,0.000712,0.403552,...,1.000000,0.990033,1.000000,1.0,0.85,1.000000,0.929293,0.948718,0.0,1.0
2,0.086758,0.093750,0.862002,0.010208,0.455103,0.893739,0.007679,0.006439,0.000972,0.437482,...,0.916667,1.000000,1.000000,1.0,1.00,1.000000,0.909091,0.974359,0.0,1.0


In [16]:
# Cell 8 — Train / Validation / Test split (70 / 15 / 15)
# Using X_scaled_df created in Cell 7

X_temp, X_test, y_temp, y_test = train_test_split(
    X_scaled_df, y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.1765,   # 0.1765 of 85% ≈ 15% of total
    random_state=42,
    stratify=y_temp
)

total = len(df)
print(f"Train set   : {X_train.shape[0]:,} samples  ({X_train.shape[0]/total*100:.1f}%)")
print(f"Validation  : {X_val.shape[0]:,}  samples  ({X_val.shape[0]/total*100:.1f}%)")
print(f"Test set    : {X_test.shape[0]:,}  samples  ({X_test.shape[0]/total*100:.1f}%)")
print(f"Total check : {X_train.shape[0]+X_val.shape[0]+X_test.shape[0]:,} (should = {total:,})")

Train set   : 41,015 samples  (70.0%)
Validation  : 8,791  samples  (15.0%)
Test set    : 8,790  samples  (15.0%)
Total check : 58,596 (should = 58,596)


In [17]:
# Cell 9 — Save all outputs
# Add labels back to scaled dataframe before saving
X_scaled_df['Class']    = y.values
X_scaled_df['Category'] = df['Category'].values

# Save cleaned scaled data
cleaned_path = os.path.join(PROC_DIR, 'cleaned_data.csv')
X_scaled_df.to_csv(cleaned_path, index=False)

# Save scaler for future use
scaler_path = os.path.join(MODEL_DIR, 'scaler.pkl')
joblib.dump(scaler, scaler_path)

print(f"Saved: cleaned_data.csv  → {cleaned_path}")
print(f"Saved: scaler.pkl        → {scaler_path}")
print(f"\nFile size: {os.path.getsize(cleaned_path)/1024/1024:.1f} MB")
print("\nNotebook 03 complete — proceed to 04_feature_engineering.ipynb")

Saved: cleaned_data.csv  → c:\Users\aryar\OneDrive\Desktop\AMRDF\AMRDF_Ransomware_Detaction\Data\processed\cleaned_data.csv
Saved: scaler.pkl        → c:\Users\aryar\OneDrive\Desktop\AMRDF\AMRDF_Ransomware_Detaction\models\scaler.pkl

File size: 49.5 MB

Notebook 03 complete — proceed to 04_feature_engineering.ipynb


In [18]:
# Cell 9 — Save all outputs
# Add labels back to scaled dataframe before saving
X_scaled_df['Class']    = y.values
X_scaled_df['Category'] = df['Category'].values

# Save cleaned scaled data
cleaned_path = os.path.join(PROC_DIR, 'cleaned_data.csv')
X_scaled_df.to_csv(cleaned_path, index=False)

# Save scaler for future use
scaler_path = os.path.join(MODEL_DIR, 'scaler.pkl')
joblib.dump(scaler, scaler_path)

print(f"Saved: cleaned_data.csv  → {cleaned_path}")
print(f"Saved: scaler.pkl        → {scaler_path}")
print(f"\nFile size: {os.path.getsize(cleaned_path)/1024/1024:.1f} MB")
print("\nNotebook 03 complete — proceed to 04_feature_engineering.ipynb")

Saved: cleaned_data.csv  → c:\Users\aryar\OneDrive\Desktop\AMRDF\AMRDF_Ransomware_Detaction\Data\processed\cleaned_data.csv
Saved: scaler.pkl        → c:\Users\aryar\OneDrive\Desktop\AMRDF\AMRDF_Ransomware_Detaction\models\scaler.pkl

File size: 49.5 MB

Notebook 03 complete — proceed to 04_feature_engineering.ipynb


In [19]:
# Cell 10 — FIXED save (preserves column names correctly)
import joblib, os

# Reset index before saving to avoid alignment issues
save_df = X_scaled_df.copy().reset_index(drop=True)
save_df['Class']    = y.reset_index(drop=True).values
save_df['Category'] = df['Category'].reset_index(drop=True).values

# Verify before saving
print(f"Shape to save        : {save_df.shape}")
print(f"NaN count            : {save_df.isnull().sum().sum()}")
print(f"Sample column names  : {save_df.columns[:5].tolist()}")
print(f"Class distribution   : \n{save_df['Class'].value_counts()}")
print(f"\nFirst row preview:")
print(save_df.head(1).T)

# Save
cleaned_path = os.path.join(PROC_DIR, 'cleaned_data.csv')
save_df.to_csv(cleaned_path, index=False)
joblib.dump(scaler, os.path.join(MODEL_DIR, 'scaler.pkl'))

print(f"\nSaved: cleaned_data.csv — {os.path.getsize(cleaned_path)/1024/1024:.1f} MB")

Shape to save        : (58596, 54)
NaN count            : 0
Sample column names  : ['pslist.nproc', 'pslist.nppid', 'pslist.avg_threads', 'pslist.avg_handlers', 'dlllist.ndlls']
Class distribution   : 
Class
Benign     29298
Malware    29298
Name: count, dtype: int64

First row preview:
                                               0
pslist.nproc                            0.109589
pslist.nppid                            0.140625
pslist.avg_threads                      0.587121
pslist.avg_handlers                     0.006766
dlllist.ndlls                           0.369275
dlllist.avg_dlls_per_proc                0.67994
handles.nhandles                        0.005379
handles.avg_handles_per_proc            0.004187
handles.nfile                           0.000501
handles.nevent                          0.316922
handles.ndesktop                        0.175182
handles.nkey                            0.181208
handles.nthread                         0.095066
handles.ndirectory        